# Colab — srd-degel (ResNet-18 entraîné en entier)

Mode d'emploi : `docs/guide_colab.md`. Protocole : `docs/protocole.md`.

1. **Exécution → Modifier le type d'exécution → GPU** (A100 ou L4 si disponible, sinon T4).
2. À **chaque session** : cellules 1 et 2 (environ 5 minutes).
3. Puis l'étape voulue. Relancer une étape coupée **reprend** là où elle s'était arrêtée.

Ne jamais lancer deux fois la même étape en même temps (un verrou refuse la seconde).

## 1. Mise en place (à chaque session)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os, subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/Arsenicophe/SRD-degel.git"
DRIVE_ROOT = Path("/content/drive/MyDrive/srd-degel")          # résultats, sur Drive
CELEBA_ZIP = Path("/content/drive/MyDrive/srd-facegeom/data/celeba/img_align_celeba.zip")  # déjà sur Drive
CODE_DIR = Path("/content/srd-degel")
RUNS = DRIVE_ROOT / "runs"
CACHE = Path("/content/cache")                                   # disque local de la session
RUNS.mkdir(parents=True, exist_ok=True)

if CODE_DIR.exists():
    subprocess.run(["git", "-C", str(CODE_DIR), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, str(CODE_DIR)], check=True)
os.chdir(CODE_DIR)
sys.path.insert(0, str(CODE_DIR / "src"))
os.environ["PYTHONPATH"] = str(CODE_DIR / "src")
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

# Seulement ce que Colab n'a pas, sans toucher à torch ni numpy (--no-deps) :
# la version qui a tourné sur Colab les 14 et 15 novembre.
%pip install -q --no-deps "pytorch-metric-learning==3.0.3"

import torch
from importlib.metadata import version
for pkg in ["torch", "torchvision", "numpy", "scikit-learn", "pytorch-metric-learning"]:
    print(f"{pkg:24}", version(pkg))
print("GPU :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "AUCUN : changer le type d'exécution")
print("Processeurs :", os.cpu_count())
C = f"--cache-dir {CACHE} --runs-dir {RUNS}"   # options communes à tous les scripts

## 2. Cache des images (à chaque session, environ 3 minutes)

Copie du zip sur le disque local, puis carrés de 128 px dans `/content/cache`.

In [ ]:
import shutil, time
local_zip = Path("/content/img_align_celeba.zip")
if not local_zip.exists() or local_zip.stat().st_size != CELEBA_ZIP.stat().st_size:
    t = time.time()
    tmp = local_zip.with_suffix(".part")
    shutil.copyfile(CELEBA_ZIP, tmp)   # copie complète puis renommage : jamais de zip tronqué
    tmp.replace(local_zip)
    print(f"zip copié en {time.time() - t:.0f} s")
!python scripts/01_build_cache.py --source {local_zip} --cache-dir {CACHE}

### Tests rapides (facultatif, environ 1 minute)

In [ ]:
!python -m pytest -q

## Étape 2 — Mesures de base

Déjà faites sur le Mac le 22 novembre : le résumé est dans le dépôt (`results/2_baselines_val.json`),
ce qui suffit pour la suite. Cette cellule ne sert que pour les refaire sur Colab.

In [ ]:
# !python scripts/02_baselines.py $C

## Étape 3 — Pilote : une époque par perte (à lancer en premier)

Mesure la durée d'une époque et vérifie que chaque perte apprend. Affiche le temps prévu
du réglage et des graines finales : **à lire avant de lancer la suite**.

In [ ]:
!python scripts/03_pilot.py $C

## Étape 4 — Réglage (10 tirages par perte, 20 époques au plus)

Une perte par session, ou `all` pour les 4 à la suite. Plusieurs sessions peuvent travailler
en parallèle sur **des pertes différentes**.

In [ ]:
LOSS = "all"   # nsoftmax, arcface, contrastive, triplet ou all
!python scripts/04_tune.py --loss {LOSS} $C

## Étape 5 — Entraînements finaux (meilleur tirage × graines 0, 1, 2)

In [ ]:
LOSS = "all"
!python scripts/05_final.py --loss {LOSS} $C

## Tout enchaîner (réglage, graines finales, évaluation sur la validation)

Pour une longue session : relancer cette cellule après une coupure reprend où le travail s'était arrêté.

In [ ]:
# Une seule ligne : chaque étape ne démarre que si la précédente a réussi
!python scripts/04_tune.py --loss all $C && python scripts/05_final.py --loss all $C && python scripts/06_evaluate.py --split val $C

## Méthodes classiques (processeur seulement, à tout moment)

In [ ]:
!python scripts/08_classical.py $C

## Étape 6 — Évaluation sur la validation et figures

In [ ]:
!python scripts/06_evaluate.py --split val $C
EVAL = sorted(RUNS.glob("*_6_eval_val"))[-1]
!python scripts/07_figures.py --eval {EVAL} --en $C

## Étape 7 — TEST, une seule fois, à la toute fin

Seulement quand tous les choix sont faits sur la validation. Le script écrit
`TEST_CONSULTED.json` dans le dossier des runs et refuse toute seconde lecture.

In [ ]:
CONFIRMER_TEST = False   # mettre True, en connaissance de cause
if CONFIRMER_TEST:
    !python scripts/06_evaluate.py --split test --confirm-test $C
    EVAL = sorted(RUNS.glob("*_6_eval_test"))[-1]
    !python scripts/07_figures.py --eval {EVAL} --en $C
else:
    print("Test non lu.")